# waterfall-py — Debt Waterfall Engine
## Demo: Midtown Mixed-Use Development Project

This notebook models a multi-tranche debt waterfall for a mixed-use real estate
development project with senior debt, mezzanine financing, and equity.

We walk through period-by-period cash distribution, DSCR tracking,
covenant breach detection, and cash sweep logic.


In [ ]:
import sys
sys.path.insert(0, '..')

from waterfall import Tranche, CashFlowPeriod, DealStructure, run
import pandas as pd
pd.set_option('display.max_columns', None)


## 1. Define the Capital Stack

In [ ]:
senior = Tranche(
    name="Senior Debt",
    principal=7_000_000,
    rate=0.06,
    term_years=10,
    priority=1,
)

mezz = Tranche(
    name="Mezzanine",
    principal=2_000_000,
    rate=0.12,
    term_years=10,
    priority=2,
    is_mezz=True,
)

equity = Tranche(
    name="Equity",
    principal=1_000_000,
    rate=0.0,
    term_years=10,
    priority=3,
    is_equity=True,
)

print(f"Total Debt:    \${(senior.principal + mezz.principal)/1e6:.1f}MM")
print(f"Total Equity:  \${equity.principal/1e6:.1f}MM")
print(f"Senior Rate:   {senior.rate*100:.1f}%")
print(f"Mezz Rate:     {mezz.rate*100:.1f}%")


## 2. Define Cash Flow Periods

We model 10 annual periods with realistic CFADS
(Cash Flow Available for Debt Service).


In [ ]:
import random
random.seed(42)

cash_flows = [
    CashFlowPeriod(
        period=i,
        cfads=round(1_100_000 + random.uniform(-100_000, 200_000)),
        operating_expenses=50_000,
    )
    for i in range(1, 11)
]

print("Period | CFADS")
print("-" * 30)
for cf in cash_flows:
    print(f"  Y{cf.period:<3}  | \${cf.cfads:,.0f}")


## 3. Define the Deal Structure

In [ ]:
deal = DealStructure(
    name="Midtown Mixed-Use Development",
    tranches=[senior, mezz, equity],
    cash_flows=cash_flows,
    min_dscr=1.25,
    dsra_months=6,
    cash_sweep_pct=1.0,
)

print(f"Total Debt:        \${deal.total_debt/1e6:.1f}MM")
print(f"Total Equity:      \${deal.total_equity/1e6:.1f}MM")
print(f"Min DSCR Covenant: {deal.min_dscr}x")
print(f"DSRA Target:       {deal.dsra_months} months")
print(f"Periods:           {deal.num_periods}")


## 4. Run the Waterfall

The waterfall distributes cash in strict priority order each period:
1. Operating expenses
2. Senior interest
3. Senior principal
4. DSRA funding
5. Mezz interest
6. Mezz principal
7. Cash sweep (excess to senior, then mezz)
8. Equity distribution (only if DSCR covenant met)


In [ ]:
result = run(deal)
df = result.summary()


## 5. DSCR Analysis

In [ ]:
dscr_df = result.dscr_table()
print(dscr_df.to_string(index=False))


## 6. Stressed Scenario — Low Cash Flow

What happens when CFADS drops 30%? We test covenant breaches
and equity lock-up under stress.


In [ ]:
stressed_flows = [
    CashFlowPeriod(period=i, cfads=800_000, operating_expenses=50_000)
    for i in range(1, 11)
]

stressed_deal = DealStructure(
    name="Midtown Mixed-Use — Stressed",
    tranches=[
        Tranche(name="Senior Debt", principal=7_000_000,
                rate=0.06, term_years=10, priority=1),
        Tranche(name="Mezzanine", principal=2_000_000,
                rate=0.12, term_years=10, priority=2, is_mezz=True),
        Tranche(name="Equity", principal=1_000_000,
                rate=0.0, term_years=10, priority=3, is_equity=True),
    ],
    cash_flows=stressed_flows,
    min_dscr=1.25,
    dsra_months=6,
    cash_sweep_pct=1.0,
)

stressed_result = run(stressed_deal)
stressed_df = stressed_result.summary()

print(f"\nCovenant Breaches: {stressed_result.num_covenant_breaches}")
print(f"Defaults:          {stressed_result.num_defaults}")
print(f"Equity Distributions: \${stressed_result.total_equity_distributions:,.0f}")


## Summary

| Scenario | Covenant Breaches | Defaults | Total Equity Dist |
|----------|------------------|----------|------------------|
| Base Case | See above | 0 | See above |
| Stressed (-30% CFADS) | Multiple | Possible | $0 (locked up) |

The waterfall engine automatically enforces priority of payments,
tracks DSCR covenants, and locks up equity distributions when
coverage ratios fall below the minimum threshold.

**GitHub:** https://github.com/Jaypatel1511/waterfall-py
**PyPI:** https://pypi.org/project/waterfall-py
